# Tutorial 5 · Uncertainty, calibration and robustness

AI in Rehabilitation and Healthcare (376-1229-00L) · HS 2026 · Lecture 6, Model evaluation and robustness

**What you'll do:**
1. Load the feature table from Tutorial 2, encode `activity`, and split **by person**.
2. Read the core classification metrics (**accuracy, precision, recall, balanced accuracy, F1**), with small examples of why one is not enough.
3. Quantify **uncertainty** with **entropy, NLL (log-loss), Brier score**, and **calibration** with **ECE/MCE** and **reliability diagrams**.
4. **Calibrate** probabilities (Platt and temperature scaling) on validation people and see how ECE and NLL change.
5. Put a **range** on a score: bootstrap the windows, then bootstrap the **people**, and compare the two intervals.
6. **Stress** the model with noise scaled to each feature and with missing values, and **choose a threshold** for one activity.

**Data.** `features_df_10s_3s.csv` is computed from the public MotionSense dataset (iPhone accelerometer and gyroscope at 50 Hz, 24 people, 6 activities) with Tutorial 2's recipe: a 10 Hz low-pass filter, then ten features per axis on 10-second windows with a 3-second step. Cite it as: Malekzadeh M, Clegg RG, Cavallaro A, Haddadi H. Mobile sensor data anonymization. *IoTDI '19*, ACM, 2019, pp. 49–58. doi:10.1145/3302505.3310068.

**With the course coach.** Run *T5.00* with your code from SCAI Learning (*Your data → Connect a notebook*). The coach then sees the code of each cell you run, and the *Report* cells send it numbers about your split, your scores and your model, never your data and never a cell's output. Ask it about a cell by its name, for example *"why does T5.05 split by person?"*. Without a code the notebook runs the same and sends nothing.


In [ ]:
# T5.00 Connect the course coach (optional)
# In SCAI Learning, open Your data → Connect a notebook and copy the six-letter code. Paste it
# between the quotes and run this cell. Left empty, nothing is installed and nothing is sent.
# Connected, the coach receives the code of each cell you run and the numbers the Report cells
# compute: never your data, never a cell's output.
COACH_CODE = ""
coach = globals().get("coach")          # connected already with the app's snippet? keep it
if COACH_CODE and coach is None:
    %pip install -q https://app.scai-learning.ch/companion.tar.gz
    import rehab_coach as coach
    coach.connect(COACH_CODE)
    coach.watch()
print("Coach connected: the cells you run from here on reach it." if coach else "Working without the coach.")


## Main Metric Summarization (what/why/how to use)
- **Accuracy**: Fraction of correct labels. *Can hide poor minority-class performance* if data are imbalanced.
- **Precision (macro)**: For each class, among predicted positives, how many are correct. *Penalizes false positives*. Macro = average across classes.
- **Recall (macro)**: For each class, among true positives, how many you found. *Penalizes false negatives*. Macro = average.
- **Balanced Accuracy**: Average of per-class recall. *Fair when classes are imbalanced*, unlike plain accuracy.
- **F1 (macro)**: Harmonic mean of precision and recall per class, then averaged. *Balances FP vs FN*.
- **Confusion Matrix**: Counts of (true, predicted) by class. *Use to see which classes are confused*.

**Probabilistic / Uncertainty / Calibration**
- **Predictive Entropy**: Uncertainty of the predicted probability vector. *High when model is unsure (probabilities spread out)*.
- **NLL / Log Loss**: Measures the difference between a model's predicted probabilities and the actual outcome. Punishes **overconfident wrong** predictions heavily. *Measures probability quality, not just argmax*.
- **Brier Score**: Squared error between predicted probabilities and one-hot truth. *Bounded & interpretable.*
- **ECE / MCE**: Expected / maximum calibration error. *How well confidences match reality*. Visualize with **reliability diagrams** (perfect = diagonal).


## 1) Setup & imports


In [ ]:
# T5.01 Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, balanced_accuracy_score,
    f1_score, confusion_matrix, log_loss
)
from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold
from sklearn.calibration import CalibratedClassifierCV
try:
    from sklearn.frozen import FrozenEstimator      # scikit-learn 1.6 and later
except ImportError:
    FrozenEstimator = None


## 2) Load the dataset


In [ ]:
# T5.02 Mount Google Drive
# Colab only. On Jupyter on your own computer, skip this cell and set data_path below to your folder.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# T5.03 Load the feature table
data_path = Path('/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/')   # Jupyter on your own computer: the folder that holds the file
load_path = data_path / 'features_df_10s_3s.csv'

features_df = pd.read_csv(load_path)
print(f"Loaded features_df from {load_path}")
print(features_df.shape)
display(features_df.head())


## 3) Prepare and process data


In [ ]:
# T5.04 Labels, features and people
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(features_df['activity'])

X = features_df.drop(['activity', 'subject', 'trial', 'window'], axis=1).to_numpy()
subjects = features_df['subject'].to_numpy()

print('X shape:', X.shape, '| #classes:', len(label_encoder.classes_))


## 4) Subject-based split (GroupShuffleSplit) and evaluation

We split by **subject** so the model is tested on **unseen people** (more realistic generalization).


In [ ]:
# T5.05 Split by person and score
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=subjects))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
subjects_train, subjects_test = subjects[train_idx], subjects[test_idx]

model = RandomForestClassifier(n_estimators=10, max_depth=None, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro')
rec = recall_score(y_test, y_pred, average='macro')
bal = balanced_accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average='macro')

test_scores = {'accuracy': acc, 'balanced_accuracy': bal, 'f1_macro': f1m}   # kept for the reports below
print(f'Test people: {len(np.unique(subjects_test))}, test windows: {len(test_idx)}')
print('=== Subject-wise Test Metrics ===')
print(f'Accuracy           : {100*acc:.2f}%  -> Overall correctness')
print(f'Precision (macro)  : {100*prec:.2f}%  -> Low if many false positives')
print(f'Recall (macro)     : {100*rec:.2f}%  -> Low if many false negatives')
print(f'Balanced Accuracy  : {100*bal:.2f}%  -> Mean per-class recall (good for imbalance)')
print(f'F1-macro           : {100*f1m:.2f}%  -> Harmonic mean of precision/recall per class')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix — Subject-wise Test")
plt.tight_layout(); plt.show()


In [ ]:
# T5.06 Report the split and its scores
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T5.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, model=model, tutorial="T5", scores=test_scores)
else:
    print("Not connected: nothing sent. Run T5.00 with your code to use the coach.")


### 4.1 Metric intuition with **mini-examples**
These toy cases show *why* we track multiple metrics.

**A) Imbalance: same accuracy, different fairness**  
We build an imbalanced set (95% class A). A dumb classifier predicting only A has **95% accuracy** but **balanced accuracy = 50%**.


In [ ]:
# T5.07 Example: imbalance
# A) Imbalance example
y_true = np.array([0]*95 + [1]*5)   # 95% class 0
y_pred_all_zero = np.array([0]*100) # predict all 0

acc = accuracy_score(y_true, y_pred_all_zero)
bal = balanced_accuracy_score(y_true, y_pred_all_zero)
print(f'Accuracy={100*acc:.1f}% vs Balanced Accuracy={100*bal:.1f}%  (reveals failure on minority class)')


**B) Same accuracy, different probability quality**  
Two models get 80% accuracy, but one is **overconfident** on its mistakes. **NLL** and **Brier** expose this.


In [ ]:
# T5.08 Example: same accuracy, different confidence
rng = np.random.RandomState(0)
n = 100
y_true = rng.randint(0, 2, size=n)

# Build equal accuracy: 80% correct predictions
correct_mask = rng.rand(n) < 0.8
y_pred_label = np.where(correct_mask, y_true, 1 - y_true)

# Model 1: Overconfident (p=0.99 for predicted label)
proba1 = np.zeros((n,2)); proba1[np.arange(n), y_pred_label] = 0.99; proba1[np.arange(n), 1-y_pred_label] = 0.01
# Model 2: Modest confidence (p=0.60 for predicted label)
proba2 = np.zeros((n,2)); proba2[np.arange(n), y_pred_label] = 0.60; proba2[np.arange(n), 1-y_pred_label] = 0.40

def brier_binary(y, p):
    Y = np.eye(2)[y]
    return np.mean(np.sum((p - Y)**2, axis=1))

def nll_binary(y, p):
    return log_loss(y, p, labels=[0,1])

print('Accuracy both:', accuracy_score(y_true, y_pred_label))
print('Overconfident — NLL:', nll_binary(y_true, proba1), 'Brier:', brier_binary(y_true, proba1))
print('Modest conf.  — NLL:', nll_binary(y_true, proba2), 'Brier:', brier_binary(y_true, proba2))
print('=> Overconfidence is punished by NLL/Brier even at same accuracy.')


**C) Entropy tracks uncertainty**  
High entropy ↔ probabilities spread out (uncertain). Low entropy ↔ one class dominates (confident).


In [ ]:
# T5.09 Example: entropy
def entropy_row(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return -np.sum(p*np.log(p), axis=1)

p_sure = np.array([[0.99, 0.01],
                   [0.98, 0.02],
                   [0.97, 0.03]])
p_unsure = np.array([[0.51, 0.49],
                     [0.55, 0.45],
                     [0.60, 0.40]])

print('Entropy (confident):', entropy_row(p_sure))
print('Entropy (uncertain):', entropy_row(p_unsure))


## 5) Uncertainty & calibration on the **subject-wise test**
We now compute uncertainty metrics on real test predictions and visualize calibration.


In [ ]:
# T5.10 Uncertainty on the test people
proba_test = model.predict_proba(X_test)

def predictive_entropy(p: np.ndarray, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log(p), axis=1)

def brier_multiclass(y_true, proba, n_classes):
    Y = np.eye(n_classes)[y_true]
    return np.mean(np.sum((proba - Y)**2, axis=1))

def ece_score(y_true, proba, n_bins=15):
    preds = proba.argmax(axis=1)
    confidences = proba.max(axis=1)
    correctness = (preds == y_true).astype(int)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    idx = np.clip(np.digitize(confidences, bins) - 1, 0, n_bins - 1)   # a confidence of exactly 1.0 belongs in the last bin
    ece, mce = 0.0, 0.0
    for b in range(n_bins):
        mask = (idx == b)
        if mask.sum() == 0: 
            continue
        acc = correctness[mask].mean()
        conf = confidences[mask].mean()
        w = mask.mean()
        ece += w * abs(acc - conf)
        mce = max(mce, abs(acc - conf))
    return ece, mce

entropy = predictive_entropy(proba_test)
try:
    nll = log_loss(y_test, proba_test, labels=np.arange(len(label_encoder.classes_)))
except Exception:
    nll = np.nan
brier = brier_multiclass(y_test, proba_test, n_classes=len(label_encoder.classes_))
ece, mce = ece_score(y_test, proba_test, n_bins=15)

print('Entropy (mean):', float(entropy.mean()))
print('NLL:', float(nll), ' — lower is better; punishes overconfident wrong predictions')
print('Brier:', float(brier), ' — lower is better; squared error of probabilities')
print('ECE:', float(ece), ' MCE:', float(mce), ' — closer to 0 is better (well-calibrated)')


In [ ]:
# T5.11 Reliability diagram
def plot_reliability_diagram(y_true, proba, n_bins=15, title='Reliability Diagram'):
    preds = proba.argmax(axis=1)
    conf = proba.max(axis=1)
    correct = (preds == y_true).astype(int)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    accs, confs = [], []
    for b in range(n_bins):
        m = (conf >= bins[b]) & (conf < bins[b+1]) if b < n_bins-1 else (conf >= bins[b]) & (conf <= bins[b+1])
        if m.sum() == 0:
            accs.append(np.nan); confs.append(np.nan)
        else:
            accs.append(correct[m].mean()); confs.append(conf[m].mean())
    plt.figure(); plt.plot([0,1],[0,1],'--', label="Ideal")
    plt.plot([c for c in confs if not np.isnan(c)], [a for a in accs if not np.isnan(a)], marker='o',label="Uncalibrated")
    plt.xlabel('Confidence'); plt.ylabel('Empirical accuracy'); plt.title(title + ' (ideal is diagonal)'); plt.legend(); plt.show()

plot_reliability_diagram(y_test, proba_test, n_bins=15, title='Reliability — Uncalibrated')
plt.figure(); plt.hist(entropy, bins=30); plt.xlabel('Entropy'); plt.ylabel('Count'); plt.title('Predictive Entropy — Test')
plt.show()


In [ ]:
# T5.12 Report the calibration
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T5.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, proba=proba_test, tutorial="T5")   # ECE, computed again by the companion
else:
    print("Not connected: nothing sent. Run T5.00 with your code to use the coach.")


## 6) What is **Model Calibration**? 

**Calibration** means: *when the model says “X% confident,” it should be right about X% of the time.*  
A perfectly calibrated classifier's confidence values match reality.

### Why it matters
- **Decision thresholds**: If 0.8 probability truly means ~80% success, you can set thresholds for deployment, triage, or human-in-the-loop.
- **Risk-sensitive apps**: Overconfident wrong predictions can be dangerous; **NLL** and **ECE** make this visible.
- **Comparability**: Two models with the same accuracy can have very different probability quality.

### Typical symptoms
- **Overconfidence**: Model predicts high probabilities but accuracy in those bins is lower.  
  *Reliability curve sits **below** the diagonal; ECE is high.*
- **Underconfidence**: Probabilities are conservative; accuracy sits **above** the curve.  
  *Reliability curve sits **above** the diagonal.*

### How we measure & visualize
- **Reliability diagram**: Bin predictions by **max probability**; for each bin plot (mean confidence, empirical accuracy).  
  *Perfectly calibrated → points on the diagonal y=x.*
- **ECE (Expected Calibration Error)**: Weighted average gap between confidence and accuracy across bins (↓ better).
- **NLL (Log loss)**: Punishes **overconfident mistakes** heavily (↓ better). Complements ECE.
- **Brier score**: Squared error between predicted probabilities and one-hot truth (↓ better).

### How to improve calibration
- **Post-hoc calibration** on a **held-out validation set**:
  - **Platt scaling (sigmoid)**: A simple logistic mapping per class (or OvR). Often effective for slight overconfidence.
  - **Temperature scaling**: Scales logits by \(1/T\); tunes **T** to minimize **NLL**. Preserves ranking, usually reduces overconfidence.
  - *(Also exists: Isotonic regression — flexible but needs more data and can overfit.)*
- **Good practice**: Use a **subject-wise** validation split (matching train/test protocol) to fit calibration, then **evaluate on the held-out test**.

*Below we fit Platt & Temperature on a subject-wise validation split and compare ECE/NLL & reliability curves before vs after.*


## 6) Calibration (Platt & Temperature) — see ECE/NLL improve
We create a **subject-wise validation split** from the training set to fit calibration and compare on the held-out test.

*A note on the comparison below.* It scores all three versions on the test people, to show what calibration does. In a study you would choose the method on validation people only and open the test set once, for the method you chose; the exercise at the end does that.


In [ ]:
# T5.13 Platt and temperature scaling
from sklearn.model_selection import GroupShuffleSplit

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, val_idx = next(gss_val.split(X_train, y_train, groups=subjects_train))
X_tr, X_val = X_train[tr_idx], X_train[val_idx]
y_tr, y_val = y_train[tr_idx], y_train[val_idx]

base_model = RandomForestClassifier(n_estimators=10, max_depth=None, n_jobs=-1, random_state=42)
base_model.fit(X_tr, y_tr)

# Fit the calibration map on the validation people only; the forest stays as it is.
if FrozenEstimator is not None:
    platt = CalibratedClassifierCV(FrozenEstimator(base_model), method='sigmoid')
else:                                              # scikit-learn before 1.6
    platt = CalibratedClassifierCV(base_model, cv='prefit', method='sigmoid')
platt.fit(X_val, y_val)

def softmax(z, axis=1):
    z = z - np.max(z, axis=axis, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=axis, keepdims=True)

def temperature_scale(proba, T):
    eps=1e-12
    logp = np.log(np.clip(proba, eps, 1.0))
    return softmax(logp / T, axis=1)

def best_T(proba, y_true, grid=(0.1, 0.15, 0.25, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0)):
    Y = np.eye(len(label_encoder.classes_))[y_true]
    best_T, best_nll = 1.0, np.inf
    for T in grid:
        pT = temperature_scale(proba, T)
        nll = -np.mean(np.sum(Y * np.log(np.clip(pT, 1e-12, 1.0)), axis=1))
        if nll < best_nll:
            best_nll, best_T = nll, T
    return best_T

proba_val_base = base_model.predict_proba(X_val)
T_opt = best_T(proba_val_base, y_val)
print('Optimal temperature (validation):', T_opt)

proba_test_base  = base_model.predict_proba(X_test)
proba_test_platt = platt.predict_proba(X_test)
proba_test_T     = temperature_scale(proba_test_base, T_opt)

def quick_ece(y_true, proba, n_bins=15):
    preds = proba.argmax(axis=1)
    confidences = proba.max(axis=1)
    correctness = (preds == y_true).astype(int)
    bins = np.linspace(0.0,1.0,n_bins+1)
    idx = np.clip(np.digitize(confidences, bins) - 1, 0, n_bins - 1)
    ece=0.0
    for b in range(n_bins):
        m=(idx==b)
        if m.sum()==0: continue
        acc=correctness[m].mean(); conf=confidences[m].mean(); w=m.mean()
        ece += w*abs(acc-conf)
    return ece

for name, p in [('Uncalibrated', proba_test_base),
                ('Platt', proba_test_platt),
                ('Temperature', proba_test_T)]:
    try:
        nll = log_loss(y_test, p, labels=np.arange(len(label_encoder.classes_)))
    except Exception:
        nll = np.nan
    ece = quick_ece(y_test, p, n_bins=15)
    print(f'{name:<12} — NLL: {nll:.4f}  |  ECE: {ece:.4f}')

def plot_rel_curves(y_true, proba_list, labels, n_bins=15, title='Calibration Comparison'):
    plt.figure()
    plt.plot([0,1],[0,1],'--', label='Ideal')
    for proba, lab in zip(proba_list, labels):
        preds = proba.argmax(axis=1)
        conf = proba.max(axis=1)
        correct = (preds == y_true).astype(int)
        bins = np.linspace(0.0, 1.0, n_bins+1)
        xs, ys = [], []
        for b in range(n_bins):
            m = (conf >= bins[b]) & (conf < bins[b+1]) if b < n_bins-1 else (conf >= bins[b]) & (conf <= bins[b+1])
            if m.sum()==0: continue
            xs.append(conf[m].mean()); ys.append(correct[m].mean())
        plt.plot(xs, ys, marker='o', label=lab)
    plt.xlabel('Confidence'); plt.ylabel('Empirical accuracy'); plt.title(title)
    plt.legend(); plt.show()

plot_rel_curves(y_test,
                [proba_test_base, proba_test_platt, proba_test_T],
                ['Uncalibrated','Platt','Temperature'],
                n_bins=15, title='Reliability Curves — Before/After Calibration')


## 7) Robustness · a range for the score

**Why:** a test score depends on which cases landed in the test set. The bootstrap redraws the test set many times and reads the spread. The first cell redraws **windows**, as most code does; the second redraws **people**. Compare the two intervals before you trust either.


In [ ]:
# T5.14 Bootstrap over windows
rng = np.random.RandomState(42)
B = 300
proba = proba_test          # the model from the split above, scored once on the test people
preds = proba.argmax(axis=1)
n_classes = proba.shape[1]
Y = np.eye(n_classes)[y_test]

def bootstrap_metrics(y_true, proba, preds, B=300, rng=None):
    rng = np.random.RandomState(42) if rng is None else rng
    accs, f1s, nlls, briers, eces = [], [], [], [], []
    for _ in range(B):
        idx = rng.randint(0, len(y_true), size=len(y_true))
        y_b = y_true[idx]
        p_b = proba[idx]
        pred_b = preds[idx]
        accs.append(accuracy_score(y_b, pred_b))
        f1s.append(f1_score(y_b, pred_b, average='macro'))
        try:
            nlls.append(log_loss(y_b, p_b, labels=np.arange(n_classes)))
        except Exception:
            nlls.append(np.nan)
        briers.append(np.mean(np.sum((p_b - Y[idx])**2, axis=1)))
        # ECE
        ece = 0.0
        conf = p_b.max(axis=1); correct = (p_b.argmax(axis=1) == y_b).astype(int)
        bins = np.linspace(0,1,16); idxb = np.clip(np.digitize(conf, bins)-1, 0, 14)
        for b in range(15):
            m = (idxb==b)
            if m.sum()==0: continue
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
        eces.append(ece)
    return pd.DataFrame({'acc':accs,'f1_macro':f1s,'nll':nlls,'brier':briers,'ece':eces})

boot = bootstrap_metrics(y_test, proba, preds, B=B, rng=rng)
summary = boot.describe(percentiles=[0.025,0.5,0.975]).T
display(summary[['mean','std','2.5%','50%','97.5%']])

plt.figure(); plt.hist(boot['acc'].dropna(), bins=30); plt.title('Bootstrap — Accuracy'); plt.xlabel('Accuracy'); plt.ylabel('Count'); plt.show()
plt.figure(); plt.hist(boot['f1_macro'].dropna(), bins=30); plt.title('Bootstrap — F1 Macro'); plt.xlabel('F1 Macro'); plt.ylabel('Count'); plt.show()
plt.figure(); plt.hist(boot['nll'].dropna(), bins=30); plt.title('Bootstrap — NLL'); plt.xlabel('NLL'); plt.ylabel('Count'); plt.show()
plt.figure(); plt.hist(boot['brier'].dropna(), bins=30); plt.title('Bootstrap — Brier'); plt.xlabel('Brier'); plt.ylabel('Count'); plt.show()
plt.figure(); plt.hist(boot['ece'].dropna(), bins=30); plt.title('Bootstrap — ECE'); plt.xlabel('ECE'); plt.ylabel('Count'); plt.show()


### 7.1 Resample people, not windows

The windows of one person are not independent: they share a body, a phone placement and a way of walking. Redrawing windows treats each as a new case, so the interval describes *these* people's windows and comes out too narrow for a new person. Redraw whole people instead: pick test subjects with replacement and take all their windows each time. The intervals widen; that width is the honest one.


In [ ]:
# T5.15 Bootstrap over people
def bootstrap_people(metric, y_true, y_hat, groups, B=1000, seed=0):
    """Resample whole people: draw subjects with replacement, keep all their windows."""
    r = np.random.default_rng(seed)
    people = np.unique(groups)
    rows = {p: np.flatnonzero(groups == p) for p in people}
    stats = []
    for _ in range(B):
        idx = np.concatenate([rows[p] for p in r.choice(people, size=len(people), replace=True)])
        stats.append(metric(y_true[idx], y_hat[idx]))
    return np.percentile(stats, [2.5, 97.5])

pred_test = proba_test.argmax(axis=1)
r_win = np.random.default_rng(0)
win = [balanced_accuracy_score(y_test[i], pred_test[i]) for i in (r_win.integers(0, len(y_test), len(y_test)) for _ in range(1000))]
lo_w, hi_w = np.percentile(win, [2.5, 97.5])
lo_p, hi_p = bootstrap_people(balanced_accuracy_score, y_test, pred_test, subjects_test)
print(f"Balanced accuracy on the test people: {balanced_accuracy_score(y_test, pred_test):.3f}")
print(f"  95% interval, resampling windows: {lo_w:.3f} – {hi_w:.3f}   (width {hi_w - lo_w:.3f})")
print(f"  95% interval, resampling people:  {lo_p:.3f} – {hi_p:.3f}   (width {hi_p - lo_p:.3f}, from {len(np.unique(subjects_test))} people)")


In [ ]:
# T5.16 Report the interval
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T5.00).
if coach:
    # the companion draws its own subject-level interval and sends it beside the score
    # the same score as before, so the test set is not opened again
    coach.interval(balanced_accuracy_score, y_test, pred_test, subjects_test, name="balanced_accuracy")
    coach.report((train_idx, test_idx), subjects, y, model=model, tutorial="T5",
                 scores={"balanced_accuracy": balanced_accuracy_score(y_test, pred_test)})
else:
    print("Not connected: nothing sent. Run T5.00 with your code to use the coach.")


## 8) Robustness · stress tests (noise and missing values)

**Why:** probe how the model reacts to small input shifts or dropouts, without retraining. Noise needs a scale: the features have very different spreads (a mean acceleration near zero, a dominant power in the thousands), so σ below is in **standard deviations of each feature**, measured on the training people. Random missing values are not a real sensor dropout either; ask what a dropout would look like in your data.


In [ ]:
# T5.17 Stress tests
# Gaussian noise scaled to each feature's spread, and random entries masked and imputed with train means.
train_means = X_train.mean(axis=0)
train_sd = X_train.std(axis=0)

def add_gaussian_noise(X, sigma=0.1, rng=None):
    """sigma is in standard deviations of each feature (train_sd), not in the features' own units."""
    rng = np.random.RandomState(42) if rng is None else rng
    return X + rng.normal(0, sigma, size=X.shape) * train_sd

def random_missing_impute(X, p=0.05, rng=None, fill_vals=None):
    rng = np.random.RandomState(42) if rng is None else rng
    Xm = X.copy().astype(float, copy=False)
    mask = rng.rand(*Xm.shape) < p
    Xm[mask] = np.nan
    if fill_vals is None:
        fill_vals = np.nanmean(Xm, axis=0)
    inds = np.where(np.isnan(Xm))
    Xm[inds] = np.take(fill_vals, inds[1])
    return Xm

def eval_metrics_on(X_t, y_true, clf):
    proba = clf.predict_proba(X_t)
    preds = proba.argmax(axis=1)
    acc = accuracy_score(y_true, preds)
    bal = balanced_accuracy_score(y_true, preds)
    f1m = f1_score(y_true, preds, average='macro')
    try:
        nll = log_loss(y_true, proba, labels=np.arange(proba.shape[1]))
    except Exception:
        nll = np.nan
    ece = quick_ece(y_true, proba, n_bins=15)
    return {'acc':acc, 'bal_acc':bal, 'f1_macro':f1m, 'nll':nll, 'ece':ece}

sigmas = [0.0, 0.1, 0.25, 0.5, 1.0]   # in SDs of each feature
noise_rows = []
for s in sigmas:
    Xn = add_gaussian_noise(X_test, sigma=s)
    noise_rows.append(dict(sigma=s, **eval_metrics_on(Xn, y_test, base_model)))
noise_df = pd.DataFrame(noise_rows)
display(noise_df)

plt.figure(); plt.plot(noise_df['sigma'], noise_df['acc'], marker='o'); plt.xlabel('Gaussian noise σ (SDs of each feature)'); plt.ylabel('Accuracy'); plt.title('Noise Robustness — Accuracy'); plt.show()
plt.figure(); plt.plot(noise_df['sigma'], noise_df['f1_macro'], marker='o'); plt.xlabel('Gaussian noise σ (SDs of each feature)'); plt.ylabel('F1 Macro'); plt.title('Noise Robustness — F1 Macro'); plt.show()
plt.figure(); plt.plot(noise_df['sigma'], noise_df['nll'], marker='o'); plt.xlabel('Gaussian noise σ (SDs of each feature)'); plt.ylabel('NLL'); plt.title('Noise Robustness — NLL'); plt.show()
plt.figure(); plt.plot(noise_df['sigma'], noise_df['ece'], marker='o'); plt.xlabel('Gaussian noise σ (SDs of each feature)'); plt.ylabel('ECE'); plt.title('Noise Robustness — ECE'); plt.show()

ps = [0.0, 0.05, 0.1, 0.2, 0.3]
miss_rows = []
for p in ps:
    Xm = random_missing_impute(X_test, p=p, fill_vals=train_means)
    miss_rows.append(dict(p=p, **eval_metrics_on(Xm, y_test, base_model)))
miss_df = pd.DataFrame(miss_rows)
display(miss_df)

plt.figure(); plt.plot(miss_df['p'], miss_df['acc'], marker='o'); plt.xlabel('Missingness p'); plt.ylabel('Accuracy'); plt.title('Missingness Robustness — Accuracy'); plt.show()
plt.figure(); plt.plot(miss_df['p'], miss_df['f1_macro'], marker='o'); plt.xlabel('Missingness p'); plt.ylabel('F1 Macro'); plt.title('Missingness Robustness — F1 Macro'); plt.show()
plt.figure(); plt.plot(miss_df['p'], miss_df['nll'], marker='o'); plt.xlabel('Missingness p'); plt.ylabel('NLL'); plt.title('Missingness Robustness — NLL'); plt.show()
plt.figure(); plt.plot(miss_df['p'], miss_df['ece'], marker='o'); plt.xlabel('Missingness p'); plt.ylabel('ECE'); plt.title('Missingness Robustness — ECE'); plt.show()

noise_drop = float(noise_df.loc[noise_df.sigma == 0.0, 'bal_acc'].iloc[0] - noise_df.loc[noise_df.sigma == 0.5, 'bal_acc'].iloc[0])
print(f"Balanced accuracy lost with noise of half a standard deviation: {noise_drop:.3f}")


In [ ]:
# T5.18 Report the stress test
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T5.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, tutorial="T5", lesson={"noise_stress_drop": noise_drop})
else:
    print("Not connected: nothing sent. Run T5.00 with your code to use the coach.")


## 9) Choosing a threshold

Every prediction above came from `argmax`: the activity with the highest probability wins. That is a threshold too, chosen by default rather than by need. Suppose one activity matters more than the others, say *downstairs* for a fall-risk study, and missing it is costly. Choose its threshold on the **validation people** for the recall you need, then read what it costs on the test people.


In [ ]:
# T5.19 Choose a threshold for one activity
target = 'downstairs'          # an activity you care about
recall_needed = 0.90           # what missing it costs decides this number, not the data
k = list(label_encoder.classes_).index(target)

p_val = platt.predict_proba(X_val)[:, k]            # calibrated probabilities on the validation people
is_val = (y_val == k)
cuts = np.linspace(0.01, 0.99, 99)
recalls = np.array([(p_val[is_val] >= c).mean() for c in cuts])
threshold = float(cuts[recalls >= recall_needed].max()) if (recalls >= recall_needed).any() else float(cuts.min())

p_te = platt.predict_proba(X_test)[:, k]
is_te = (y_test == k)
for name, flag in [('argmax', platt.predict(X_test) == k), (f'threshold {threshold:.2f}', p_te >= threshold)]:
    tp, fp, fn = (flag & is_te).sum(), (flag & ~is_te).sum(), (~flag & is_te).sum()
    print(f"{name:>15}: recall {tp / max(tp + fn, 1):.2f}, precision {tp / max(tp + fp, 1):.2f}, "
          f"{fn} missed and {fp} false alarms among {len(y_test)} test windows")


In [ ]:
# T5.20 Report the threshold
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T5.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, tutorial="T5", lesson={"threshold_in_use": threshold})
else:
    print("Not connected: nothing sent. Run T5.00 with your code to use the coach.")


## 10) Takeaways
- **Accuracy** alone can mislead on imbalanced data → check **balanced accuracy**, **per-class recall**, and the **confusion matrix**.
- **NLL** and **Brier** capture **probability quality** — they expose overconfidence even when accuracy is the same.
- **ECE/MCE** + **reliability diagrams** tell you if “70% confident” really means “~70% correct.”
- **Calibration** (Platt / temperature) often reduces ECE and NLL without changing accuracy.
- **Bootstrap** intervals quantify metric uncertainty, and only resampling **people** describes new people; **stress tests** need a scale and reveal brittleness to noise and missingness.
- **Thresholds** are choices: set them on validation people for the error you can afford, then check them once on the test people.


# Exercise

Can you do cross-validation for the temperature finetuning in model calibration? And show the (possible) improvement of calibrated prediction on the test set.

Keep the test people out of the loop: choose the temperature with grouped cross-validation (`GroupKFold`) inside the training people only, then score the test people once. If you are connected, point the coach at your cell and ask it where the test set enters.
